# Property Lookup — sale history by address or block/lot

Find every recorded sale of a specific property (or set of properties) from `data/merged.csv`.

- **By address** — case-insensitive, matched on word boundaries (so `4 Blackstone` won't catch `14 Blackstone`). Addresses are unnormalized (`Ridge Dr` vs `Ridge Drive`), so include the suffix only if you know which spelling the record uses.
- **By block/lot** — exact parcel key. `Qual` distinguishes condo units that share one block/lot.

A parcel re-sold over the years shows up as multiple rows — that's the sale history. `history()` operates on a single parcel and refuses a multi-parcel result.

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_colwidth", 40)
DATA = Path("..") / "data" / "merged.csv"

df = pd.read_csv(DATA, dtype=str)
df["sale_date"] = pd.to_datetime(df["Sale Date"], format="%m/%d/%Y", errors="coerce")
df["price"] = pd.to_numeric(df["Sale Price"], errors="coerce")
# normalized block/lot/qual for matching (strip whitespace; treat missing Qual as "")
for c in ("Block", "Lot", "Qual"):
    df[c.lower()] = df[c].fillna("").str.strip()
df["parcel"] = df["block"] + "_" + df["lot"] + ("_" + df["qual"]).where(df["qual"] != "", "")

SHOW = ["sale_date", "price", "Property Address", "Block", "Lot", "Qual",
        "Buyer Name", "Seller Name", "Beds", "Baths", "Sq Ft", "Year Built", "Property URL"]
print(f"loaded {len(df):,} sales")

loaded 20,309 sales


## Helpers

In [ ]:
import re

def by_address(query):
    """All sales whose address matches `query` (case-insensitive), newest first.

    Matches on word boundaries, so `\"4 Blackstone\"` returns only 4 Blackstone —
    not 14/24/34 Blackstone. Use a fuller query (`\"4 Blackstone Dr\"`) to be exact.
    """
    pat = r"\b" + re.escape(query.strip()) + r"\b"
    m = df[df["Property Address"].str.contains(pat, case=False, na=False, regex=True)]
    return m.sort_values("sale_date", ascending=False)[SHOW].reset_index(drop=True)

def by_block_lot(block, lot=None, qual=None):
    """All sales for a block (and optional lot / qual), newest first."""
    m = df[df["block"] == str(block).strip()]
    if lot is not None:
        m = m[m["lot"] == str(lot).strip()]
    if qual is not None:
        m = m[m["qual"] == str(qual).strip()]
    return m.sort_values("sale_date", ascending=False)[SHOW].reset_index(drop=True)

def history(matches):
    """Sale-over-sale appreciation between consecutive arms-length sales of ONE parcel.

    Refuses if `matches` spans multiple parcels (block/lot/qual) — chaining sales
    of different houses would produce meaningless appreciation. Narrow the query,
    or pass a single parcel via by_block_lot(...).
    """
    parcels = (matches["Block"].fillna("") + "_" + matches["Lot"].fillna("")
               + "_" + matches["Qual"].fillna("")).unique()
    if len(parcels) > 1:
        addrs = ", ".join(sorted(matches["Property Address"].unique()))
        raise ValueError(
            f"history() expects one parcel but got {len(parcels)}: {addrs}. "
            "Narrow the address query or use by_block_lot(block, lot, qual)."
        )
    h = matches.sort_values("sale_date")[["sale_date", "price", "Property Address", "Buyer Name", "Seller Name"]].copy()
    arms = h[h["price"] >= 1000].copy()
    arms["prev_price"] = arms["price"].shift()
    arms["prev_date"]  = arms["sale_date"].shift()
    arms["years_held"] = (arms["sale_date"] - arms["prev_date"]).dt.days / 365.25
    arms["change_%"]   = (arms["price"] / arms["prev_price"] - 1) * 100
    arms["cagr_%"]     = ((arms["price"] / arms["prev_price"]) ** (1 / arms["years_held"]) - 1) * 100
    return arms[["sale_date", "price", "years_held", "change_%", "cagr_%", "Buyer Name"]].round(1).reset_index(drop=True)

## Search by address

Matches on word boundaries: `"4 Blackstone"` returns **only** 4 Blackstone, not 14/24/34 Blackstone. Add the street (`"82 Ridge Dr"`) to disambiguate, or query just the street name (`"Ridge Dr"`) to list every house on it.

In [3]:
ADDRESS = "4 Blackstone"

res = by_address(ADDRESS)
print(f"{len(res)} sale(s) matching '{ADDRESS}'")
res

12 sale(s) matching '4 Blackstone'


,sale_date,price,Property Address,Block,Lot,Qual,Buyer Name,Seller Name,Beds,Baths,Sq Ft,Year Built,Property URL
0,2025-07-19,1400000.0,14 Blackstone Dr,00601,00010,NaN,"SAJIMON, SHEENA","TOKAR, ALINA & VAYSMAN, PAVEL",4,2.1,2231,1955,https://stateinfoservice.com/propert...
1,2024-12-12,1180000.0,14 Blackstone Dr,00601,00010,NaN,"TOKAR, ALINA & VAYSMAN, PAVEL","BRIGHT, JONATHAN K. (CO TRUSTEE)",4,2.1,2231,1955,https://stateinfoservice.com/propert...
2,2021-10-19,615000.0,4 Blackstone Drive,601,15,NaN,4 BLACKSTONE LLC,"VILLANUEVA, LEONARDO S & JOSEFINA",4,2.0,2210,1955,https://stateinfoservice.com/propert...
3,2021-07-14,1777000.0,24 Blackstone Drive,401,47,NaN,"GLASSMAN, ISAAC & BROOKE",RAINBOW RIDGE HOMES LLC,6,4.1,4462,2018,https://stateinfoservice.com/propert...
4,2018-07-27,1325000.0,34 Blackstone Dr,401,72,NaN,"FELDMAN, DAVID & GRACE","GIVNER, MICHAEL & ALISON",4,3.0,4220,1966,https://stateinfoservice.com/propert...
5,2011-09-27,560000.0,24 Blackstone Drive,401,47,NaN,"RAINBOW RIDGE HOMES,LLC","EHRLICH, JOSEPH HERBER",6,4.1,4462,2018,https://stateinfoservice.com/propert...
6,2008-03-28,1200000.0,34 Blackstone Dr,401,72,NaN,"GIVNER, MICHAEL & ALISON","MAJESTIC HOMES LLC,",4,3.0,4220,1966,https://stateinfoservice.com/propert...
7,2007-08-31,675000.0,34 Blackstone Dr,401,72,NaN,"MAJESTIC HOMES LLC,","CHU, JOHN",4,3.0,4220,1966,https://stateinfoservice.com/propert...
8,2006-11-03,665000.0,44 Blackstone Drive,401,67,NaN,"ZACHEO, ANTONIO & SONIA","LEE, CHONG-WON & SOON",4,2.1,2465,1966,https://stateinfoservice.com/propert...
9,2006-02-27,1.0,14 Blackstone Dr,601,10,NaN,"BRIGHT, MAURICE & HELENE","BRIGHT, MAURICE",4,2.1,2231,1955,https://stateinfoservice.com/propert...


## Search by block / lot

`lot` and `qual` are optional. Omit `lot` to see a whole block; pass `qual` to pin a single condo unit.

In [ ]:
BLOCK = "601"
LOT = "15"
QUAL = None

res = by_block_lot(BLOCK, LOT, QUAL)
print(f"{len(res)} sale(s) for block {BLOCK} lot {LOT} qual {QUAL}")
res

3 sale(s) for block 5001 lot 38 qual None


,sale_date,price,Property Address,Block,Lot,Qual,Buyer Name,Seller Name,Beds,Baths,Sq Ft,Year Built,Property URL
0,2023-08-31,845000.0,82 Ridge Dr,5001,38,NaN,"FU, LITAO","KORNBERG, ANAT & ARIEL",3,2.1,1838,1962,https://stateinfoservice.com/propert...
1,2005-08-12,615000.0,82 Ridge Dr,5001,38,NaN,"KORNBERG, ANAT & ARIEL","LERMAN, CRAIG & SUSETTE",3,2.1,1838,1962,https://stateinfoservice.com/propert...
2,2001-05-16,1.0,82 Ridge Dr,5001,38,NaN,"LERMAN, CRAIG & SUSETTE","LERMAN, CRAIG & SUSETTE",3,2.1,1838,1962,https://stateinfoservice.com/propert...


## Sale-to-sale history

Appreciation between consecutive arms-length sales (price ≥ $1,000) of whatever `res` currently holds. For a single parcel this is its ownership/price history; for a street it mixes properties, so prefer a single parcel here.

In [5]:
history(res)

,sale_date,price,years_held,change_%,cagr_%,Buyer Name
0,2005-08-12,615000.0,NaN,NaN,NaN,"KORNBERG, ANAT & ARIEL"
1,2023-08-31,845000.0,18.1,37.4,1.8,"FU, LITAO"
